# EDA Completo — Dataset Enriquecido — Sistema de Recomendação (Sazonalidade + Categorias + Genero)

Análise exploratória do dataset sintético de interações de e-commerce.

Pergunta central: **o dataset gerado tem as propriedades estatísticas necessárias para treinar um modelo de recomendação confiável?**

0. Business Understanding
1. Carregamento e contexto da base
2. Qualidade estrutural dos dados
3. Distribuição de interações e funil de engajamento
4. Popularidade de itens — cauda longa
5. Atividade dos usuários
6. Esparsidade da matriz usuário-item
7. Análise temporal
8. Data readiness para recomendação
9. Conclusão executiva

## 0) Business Understanding

### Problema de negócio
Uma plataforma de e-commerce quer aumentar conversão e ticket médio recomendando produtos relevantes com base no histórico de navegação. A decisão prática é: **quais produtos apresentar a cada usuário para maximizar a probabilidade de compra?**

### Objetivo de negócio
- Aumentar a taxa de conversão (view → purchase)
- Ampliar o ticket médio via cross-sell e up-sell relevante
- Reduzir o tempo de descoberta de produto

### Stakeholders
| Stakeholder | Papel |
|---|---|
| Produto / Catálogo | Define regras de negócio e restrições de exibição |
| Marketing | Usa recomendações em campanhas e e-mails |
| Engenharia de dados | Mantém pipeline de ingestão e features |
| Data Science / MLOps | Treina, avalia e monitora o modelo |
| Liderança | Avalia impacto em receita e NPS |

### Métricas de sucesso
**Negócio:** taxa de clique nas recomendações (CTR), taxa de conversão, receita incremental  
**Modelo:** Precision@K, Recall@K, NDCG@K, Coverage@K (K=10)

### Custo do erro
| Tipo | Descrição | Consequência | Severidade |
|---|---|---|---|
| **Falso Negativo** | Item relevante não recomendado | Oportunidade de venda perdida | Alta |
| **Falso Positivo** | Item irrelevante recomendado | Experiência ruim, usuário ignora | Média |

### Riscos identificados
- **Popularity bias:** modelo pode concentrar recomendações nos itens mais populares
- **Cold start:** usuários com poucas interações têm embeddings ruins
- **Esparsidade:** matriz usuário-item muito esparsa dificulta generalização
- **Dataset sintético:** distribuições controladas podem não capturar comportamentos reais

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("../data/raw/interactions_enriched.parquet")
RANDOM_STATE = 42
TOP_K = 10
MIN_INTERACTIONS_USER = 5

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado: {DATA_PATH.resolve()}\n"
        "Execute: poetry run python scripts/generate_dataset.py"
    )

## 1) Carregamento e contexto da base

Verificamos volumetria, tipos de variáveis e uma amostra dos dados.

In [ ]:
df = pd.read_parquet(DATA_PATH)

print(f"Arquivo : {DATA_PATH}")
print(f"Linhas  : {len(df):,}")
print(f"Colunas : {df.shape[1]}")
print(f"Período : {df['timestamp'].min()} → {df['timestamp'].max()}")
print()
display(df.head(8))

In [ ]:
overview = pd.DataFrame({
    "metrica": [
        "total_interacoes", "usuarios_unicos", "itens_unicos",
        "periodo_dias", "interacoes_por_usuario_media",
        "interacoes_por_item_media",
    ],
    "valor": [
        len(df),
        df["user_id"].nunique(),
        df["item_id"].nunique(),
        (df["timestamp"].max() - df["timestamp"].min()).days,
        round(len(df) / df["user_id"].nunique(), 2),
        round(len(df) / df["item_id"].nunique(), 2),
    ],
})
display(overview)

## 2) Qualidade estrutural dos dados

Checamos duplicidades, missing values e consistência de tipos antes de qualquer análise.

In [ ]:
quality_flags = pd.DataFrame({
    "check": [
        "Linhas duplicadas",
        "Missing em user_id",
        "Missing em item_id",
        "Missing em interaction_type",
        "Missing em timestamp",
        "interaction_type fora do domínio",
    ],
    "resultado": [
        int(df.duplicated().sum()),
        int(df["user_id"].isna().sum()),
        int(df["item_id"].isna().sum()),
        int(df["interaction_type"].isna().sum()),
        int(df["timestamp"].isna().sum()),
        int(~df["interaction_type"].isin(["view", "add_to_cart", "purchase"]).sum()),
    ],
})
quality_flags["status"] = quality_flags["resultado"].apply(
    lambda v: "✓ OK" if v == 0 else "⚠ Atenção"
)
display(quality_flags)

In [ ]:
schema = pd.DataFrame({
    "coluna": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing": df.isna().sum().values,
    "n_unique": df.nunique().values,
    "exemplo": [df[c].iloc[0] for c in df.columns],
})
display(schema)

In [ ]:
# Validacao das colunas novas do dataset enriquecido
new_cols_summary = pd.DataFrame({
    "coluna": ["category", "user_gender"],
    "n_unique": [df["category"].nunique(), df["user_gender"].nunique()],
    "valores": [
        sorted(df["category"].unique().tolist()),
        sorted(df["user_gender"].unique().tolist()),
    ],
    "missing": [df["category"].isna().sum(), df["user_gender"].isna().sum()],
})
display(new_cols_summary)

## 3) Distribuição de interações e funil de engajamento

Analisamos o volume de cada tipo de evento e o funil **view → add_to_cart → purchase**, essencial para entender a intensidade do sinal de preferência.

In [ ]:
event_counts = (
    df["interaction_type"]
    .value_counts()
    .rename_axis("evento")
    .reset_index(name="contagem")
)
event_counts["pct"] = event_counts["contagem"] / len(df) * 100
display(event_counts)

In [ ]:
EVENTS = ["view", "add_to_cart", "purchase"]
LABELS = ["View", "Add to Cart", "Purchase"]
COLORS = ["#4C72B0", "#DD8452", "#55A868"]

funnel_users = [
    df[df["interaction_type"] == e]["user_id"].nunique()
    for e in EVENTS
]
funnel_interactions = [
    int(event_counts.loc[event_counts["evento"] == e, "contagem"].values[0])
    for e in EVENTS
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume de interações por tipo
axes[0].bar(LABELS, funnel_interactions, color=COLORS)
for i, v in enumerate(funnel_interactions):
    axes[0].text(i, v + 200, f"{v:,}\n({v/sum(funnel_interactions)*100:.1f}%)",
                 ha="center", fontsize=10)
axes[0].set_title("Volume de interações por tipo")
axes[0].set_ylabel("Contagem")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Usuários únicos por nível do funil
axes[1].bar(LABELS, funnel_users, color=COLORS)
for i, v in enumerate(funnel_users):
    pct = v / funnel_users[0] * 100
    axes[1].text(i, v + 10, f"{v:,}\n({pct:.1f}%)", ha="center", fontsize=10)
axes[1].set_title("Usuários únicos por nível do funil")
axes[1].set_ylabel("Usuários únicos")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.suptitle("Funil de Engajamento", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**Interpretação do funil:**  
A taxa de conversão view → purchase revela a qualidade do sinal implícito. Quanto menor a conversão, mais ruidoso o sinal de preferência — o modelo precisa aprender a separar curiosidade de intenção de compra.

### 3.1) Funil e volume por categoria de produto

Categorias diferentes têm padrões de compra distintos. Entender isso orienta a estratégia de recomendação e a análise de viés (popularity bias por categoria).

In [ ]:
cat_volume = df.groupby("category").size().rename("total").reset_index()
cat_purchase = (
    df[df["interaction_type"] == "purchase"]
    .groupby("category").size()
    .div(df.groupby("category").size())
    .mul(100).round(2)
    .rename("purchase_rate_pct")
    .reset_index()
)
cat_summary = cat_volume.merge(cat_purchase, on="category")
cat_summary["pct_interacoes"] = (cat_summary["total"] / len(df) * 100).round(1)
display(cat_summary.sort_values("total", ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=cat_summary.sort_values("total", ascending=False),
            x="category", y="total", ax=axes[0], palette="deep")
axes[0].set_title("Volume de interacoes por categoria")
axes[0].set_xlabel("Categoria")
axes[0].set_ylabel("Interacoes")
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
)

sns.barplot(data=cat_summary.sort_values("purchase_rate_pct", ascending=False),
            x="category", y="purchase_rate_pct", ax=axes[1], palette="deep")
for i, row in cat_summary.sort_values(
        "purchase_rate_pct", ascending=False).reset_index().iterrows():
    axes[1].text(i, row["purchase_rate_pct"] + 0.05,
                 f"{row['purchase_rate_pct']:.1f}%", ha="center", fontsize=10)
axes[1].set_title("Taxa de purchase por categoria (%)")
axes[1].set_xlabel("Categoria")
axes[1].set_ylabel("% purchases")

plt.suptitle("Analise por Categoria", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 4) Popularidade de itens — cauda longa

Sistemas de recomendação reais seguem uma **lei de potência (distribuição de Zipf)**: poucos itens concentram a maioria das interações. Aqui verificamos se o dataset sintético replica essa propriedade.

In [ ]:
item_counts = (
    df.groupby("item_id")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="n_interactions")
)
item_counts["rank"] = range(1, len(item_counts) + 1)
item_counts["pct_cumulative"] = (
    item_counts["n_interactions"].cumsum() / item_counts["n_interactions"].sum() * 100
)

# Percentual de interações concentradas nos top-10%
n_top10 = max(1, int(len(item_counts) * 0.10))
share_top10 = item_counts.head(n_top10)["n_interactions"].sum() / item_counts["n_interactions"].sum() * 100
n_top20 = max(1, int(len(item_counts) * 0.20))
share_top20 = item_counts.head(n_top20)["n_interactions"].sum() / item_counts["n_interactions"].sum() * 100

concentration = pd.DataFrame({
    "grupo": ["Top 10% itens", "Top 20% itens", "Restante 80% itens"],
    "pct_interacoes": [round(share_top10, 1), round(share_top20, 1), round(100 - share_top20, 1)],
})
display(concentration)

print(f"Item mais popular : {item_counts.iloc[0]['n_interactions']:,} interações")
print(f"Item menos popular: {item_counts.iloc[-1]['n_interactions']:,} interações")
print(f"Mediana por item  : {item_counts['n_interactions'].median():.0f} interações")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribuição de popularidade (histograma)
axes[0].hist(item_counts["n_interactions"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].set_title("Distribuição de interações por item")
axes[0].set_xlabel("Número de interações")
axes[0].set_ylabel("Contagem de itens")
axes[0].axvline(item_counts["n_interactions"].median(), color="red",
                linestyle="--", label=f"Mediana = {item_counts['n_interactions'].median():.0f}")
axes[0].legend()

# Curva de Lorenz (concentração)
axes[1].plot(item_counts["rank"], item_counts["pct_cumulative"],
             color="#4C72B0", linewidth=2)
axes[1].axhline(50, color="gray", linestyle=":", alpha=0.7)
axes[1].axvline(n_top10, color="#DD8452", linestyle="--",
                label=f"Top 10% = {share_top10:.1f}% das interações")
axes[1].axvline(n_top20, color="#55A868", linestyle="--",
                label=f"Top 20% = {share_top20:.1f}% das interações")
axes[1].set_title("Distribuição acumulada de interações (cauda longa)")
axes[1].set_xlabel("Rank do item (mais popular → menos popular)")
axes[1].set_ylabel("% acumulada das interações")
axes[1].legend()

plt.suptitle("Popularidade de Itens", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Verifica lei de potência em escala log-log
fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(item_counts["rank"], item_counts["n_interactions"],
          "o", markersize=3, alpha=0.5, color="#4C72B0", label="Itens")

# Ajuste linear em log-log (lei de potência)
log_rank = np.log(item_counts["rank"])
log_count = np.log(item_counts["n_interactions"])
coeffs = np.polyfit(log_rank, log_count, 1)
fit_line = np.exp(np.polyval(coeffs, log_rank))
ax.loglog(item_counts["rank"], fit_line, "--", color="red",
          linewidth=2, label=f"Ajuste lei de potência (α={-coeffs[0]:.2f})")

ax.set_title("Verificação de lei de potência — escala log-log")
ax.set_xlabel("Rank do item (log)")
ax.set_ylabel("Interações (log)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Expoente da lei de potência (α): {-coeffs[0]:.4f}")
print("(α ≈ 1 indica distribuição de Zipf clássica)")

## 5) Atividade dos usuários

A distribuição de atividade por usuário define **quem tem sinal suficiente para o modelo aprender** e **qual é o problema de cold start**.

In [ ]:
user_counts = (
    df.groupby("user_id")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="n_interactions")
)

# Percentis de atividade
percentiles = [10, 25, 50, 75, 90, 95, 99]
activity_summary = pd.DataFrame({
    "percentil": [f"p{p}" for p in percentiles],
    "n_interactions": [int(np.percentile(user_counts["n_interactions"], p)) for p in percentiles],
})
display(activity_summary)

# Usuários por faixa de atividade
bins = [0, 5, 15, 30, 60, float("inf")]
labels = ["1-5", "6-15", "16-30", "31-60", "61+"]
user_counts["faixa"] = pd.cut(user_counts["n_interactions"], bins=bins, labels=labels)
faixa_summary = (
    user_counts.groupby("faixa", observed=True)
    .size()
    .rename_axis("faixa_interacoes")
    .reset_index(name="n_usuarios")
)
faixa_summary["pct"] = faixa_summary["n_usuarios"] / len(user_counts) * 100
display(faixa_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de atividade
axes[0].hist(user_counts["n_interactions"], bins=40,
             color="#4C72B0", edgecolor="white")
axes[0].axvline(MIN_INTERACTIONS_USER, color="red", linestyle="--",
                label=f"Mínimo para treino ({MIN_INTERACTIONS_USER} interações)")
axes[0].axvline(user_counts["n_interactions"].median(), color="orange",
                linestyle="--",
                label=f"Mediana = {user_counts['n_interactions'].median():.0f}")
axes[0].set_title("Distribuição de interações por usuário")
axes[0].set_xlabel("Número de interações")
axes[0].set_ylabel("Número de usuários")
axes[0].legend(fontsize=9)

# Usuários por faixa de atividade
sns.barplot(data=faixa_summary, x="faixa_interacoes", y="n_usuarios",
            ax=axes[1], color="#4C72B0")
for i, row in faixa_summary.iterrows():
    axes[1].text(i, row["n_usuarios"] + 5, f"{row['pct']:.1f}%",
                 ha="center", fontsize=10)
axes[1].set_title("Usuários por faixa de atividade")
axes[1].set_xlabel("Interações por usuário")
axes[1].set_ylabel("Número de usuários")

plt.suptitle("Atividade dos Usuários", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

cold_users = int((user_counts["n_interactions"] < MIN_INTERACTIONS_USER).sum())
print(f"Usuários com < {MIN_INTERACTIONS_USER} interações (cold start): "
      f"{cold_users:,} ({cold_users/len(user_counts)*100:.1f}%)")

### 5.1) Distribuicao por genero

Genero e um atributo sensivel relevante para analise de fairness: queremos garantir que o modelo nao beneficia sistematicamente usuarios de um genero em detrimento de outro.

In [ ]:
gender_counts = df["user_gender"].value_counts().reset_index()
gender_counts.columns = ["genero", "interacoes"]
gender_counts["pct"] = (gender_counts["interacoes"] / len(df) * 100).round(1)

users_per_gender = (
    df.groupby("user_gender")["user_id"]
    .nunique()
    .rename("usuarios_unicos")
    .reset_index()
    .rename(columns={"user_gender": "genero"})
)
gender_purchase = (
    df[df["interaction_type"] == "purchase"]
    .groupby("user_gender").size()
    .div(df.groupby("user_gender").size())
    .mul(100).round(2)
    .rename("purchase_rate_pct")
    .reset_index()
    .rename(columns={"user_gender": "genero"})
)
gender_summary = gender_counts.merge(users_per_gender, on="genero").merge(
    gender_purchase, on="genero"
)
display(gender_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
pal = ["#4C72B0", "#DD8452", "#55A868"]

sns.barplot(data=gender_counts, x="genero", y="interacoes",
            ax=axes[0], palette=pal)
for i, row in gender_counts.iterrows():
    axes[0].text(i, row["interacoes"] + 100,
                 f"{row['pct']:.1f}%", ha="center", fontsize=10)
axes[0].set_title("Interacoes por genero")
axes[0].set_xlabel("Genero")
axes[0].set_ylabel("Interacoes")

sns.barplot(data=gender_purchase, x="genero", y="purchase_rate_pct",
            ax=axes[1], palette=pal)
for i, row in gender_purchase.iterrows():
    axes[1].text(i, row["purchase_rate_pct"] + 0.05,
                 f"{row['purchase_rate_pct']:.1f}%", ha="center", fontsize=10)
axes[1].set_title("Taxa de purchase por genero")
axes[1].set_xlabel("Genero")
axes[1].set_ylabel("% purchases")

plt.suptitle("Analise por Genero", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**Interpretacao de fairness:**  
Se a taxa de purchase diferir significativamente entre generos, pode indicar vies no dataset. O Model Card deve documentar essa disparidade e seus possiveis impactos.

## 6) Esparsidade da matriz usuário-item

A esparsidade é a principal característica estrutural que define a dificuldade do problema de recomendação. Matrizes muito esparsas (>99%) tornam métodos baseados em memória ineficazes e justificam abordagens baseadas em embeddings.

In [ ]:
n_users = df["user_id"].nunique()
n_items = df["item_id"].nunique()
n_interactions = len(df)
n_unique_pairs = df[["user_id", "item_id"]].drop_duplicates().shape[0]
sparsity = 1 - n_unique_pairs / (n_users * n_items)

matrix_summary = pd.DataFrame({
    "metrica": [
        "Usuários únicos",
        "Itens únicos",
        "Interações totais",
        "Pares (usuário, item) únicos",
        "Tamanho máximo da matriz",
        "Esparsidade",
        "Densidade",
    ],
    "valor": [
        f"{n_users:,}",
        f"{n_items:,}",
        f"{n_interactions:,}",
        f"{n_unique_pairs:,}",
        f"{n_users * n_items:,}",
        f"{sparsity:.4%}",
        f"{1 - sparsity:.4%}",
    ],
})
display(matrix_summary)

In [ ]:
# Heatmap amostral da matriz (amostra de 50 usuários x 50 itens)
sample_users = df["user_id"].value_counts().head(50).index.tolist()
sample_items = df["item_id"].value_counts().head(50).index.tolist()

df_sample = df[
    df["user_id"].isin(sample_users) & df["item_id"].isin(sample_items)
]
matrix_sample = (
    df_sample.groupby(["user_id", "item_id"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    matrix_sample.clip(upper=5),
    ax=ax,
    cmap="Blues",
    cbar_kws={"label": "Interações (máx=5)"},
    xticklabels=False,
    yticklabels=False,
)
ax.set_title(
    "Amostra da matriz usuário-item\n"
    "(top-50 usuários mais ativos × top-50 itens mais populares)",
    fontsize=12,
)
ax.set_xlabel("Itens")
ax.set_ylabel("Usuários")
plt.tight_layout()
plt.show()

sample_sparsity = 1 - (matrix_sample > 0).values.sum() / matrix_sample.size
print(f"Esparsidade na amostra (top-50×50): {sample_sparsity:.4%}")

## 7) Análise temporal

Entender a distribuição temporal é crítico para o **split de treino/teste**: em recomendação, o split deve ser **temporal** (treinar no passado, avaliar no futuro), não aleatório — caso contrário há vazamento de informação.

In [ ]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()

daily_counts = df.groupby("date").size().reset_index(name="interacoes")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Interações ao longo do tempo
axes[0, 0].plot(daily_counts["date"], daily_counts["interacoes"],
               color="#4C72B0", linewidth=1.5)
axes[0, 0].fill_between(daily_counts["date"], daily_counts["interacoes"],
                        alpha=0.2, color="#4C72B0")
axes[0, 0].set_title("Interações por dia")
axes[0, 0].set_xlabel("Data")
axes[0, 0].set_ylabel("Interações")
axes[0, 0].tick_params(axis="x", rotation=30)

# Distribuição por hora do dia
hour_counts = df.groupby("hour").size().reset_index(name="interacoes")
axes[0, 1].bar(hour_counts["hour"], hour_counts["interacoes"], color="#4C72B0")
axes[0, 1].set_title("Interações por hora do dia")
axes[0, 1].set_xlabel("Hora")
axes[0, 1].set_ylabel("Interações")
axes[0, 1].set_xticks(range(0, 24))

# Distribuição por tipo de evento ao longo do tempo
for event, color in zip(["view", "add_to_cart", "purchase"],
                        ["#4C72B0", "#DD8452", "#55A868"]):
    daily_event = (
        df[df["interaction_type"] == event]
        .groupby("date")
        .size()
        .reset_index(name="interacoes")
    )
    axes[1, 0].plot(daily_event["date"], daily_event["interacoes"],
                   label=event, color=color, linewidth=1.5)
axes[1, 0].set_title("Interações por tipo ao longo do tempo")
axes[1, 0].set_xlabel("Data")
axes[1, 0].set_ylabel("Interações")
axes[1, 0].tick_params(axis="x", rotation=30)
axes[1, 0].legend()

# Distribuição por dia da semana
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]
dow_counts = (
    df.groupby("day_of_week")
    .size()
    .reindex(day_order)
    .reset_index(name="interacoes")
)
axes[1, 1].bar(range(7), dow_counts["interacoes"], color="#4C72B0")
axes[1, 1].set_title("Interações por dia da semana")
axes[1, 1].set_xlabel("Dia")
axes[1, 1].set_ylabel("Interações")
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(
    ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"], fontsize=9
)

plt.suptitle("Análise Temporal", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Ponto de corte para split temporal (80% treino / 20% teste)
split_pct = 0.80
cutoff_idx = int(len(df) * split_pct)
df_sorted = df.sort_values("timestamp")
cutoff_date = df_sorted.iloc[cutoff_idx]["timestamp"]

train_size = int((df["timestamp"] < cutoff_date).sum())
test_size  = int((df["timestamp"] >= cutoff_date).sum())

split_summary = pd.DataFrame({
    "conjunto": ["Treino", "Teste"],
    "data_inicio": [
        str(df["timestamp"].min().date()),
        str(cutoff_date.date()),
    ],
    "data_fim": [
        str(cutoff_date.date()),
        str(df["timestamp"].max().date()),
    ],
    "n_interacoes": [train_size, test_size],
    "pct": [f"{train_size/len(df)*100:.1f}%", f"{test_size/len(df)*100:.1f}%"],
})
display(split_summary)
print(f"\nData de corte: {cutoff_date}")

### 7.1) Sazonalidade semanal

O gerador enriquecido concentra 2.5x mais trafego nos fins de semana com funil diferente (mais purchase/add_to_cart). Aqui confirmamos que o padrao foi gerado corretamente.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]
day_labels = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sab", "Dom"]

dow_volume = (
    df.groupby("day_of_week").size()
    .reindex(day_order).reset_index(name="interacoes")
)
dow_purchase = (
    df[df["interaction_type"] == "purchase"]
    .groupby("day_of_week").size()
    .div(df.groupby("day_of_week").size())
    .reindex(day_order).mul(100).reset_index(name="purchase_rate")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#DD8452" if d in ["Saturday", "Sunday"] else "#4C72B0"
          for d in day_order]

axes[0].bar(day_labels, dow_volume["interacoes"], color=colors)
axes[0].set_title("Interacoes por dia da semana")
axes[0].set_ylabel("Interacoes")
axes[0].set_xlabel("Dia (laranja = fim de semana)")

axes[1].bar(day_labels, dow_purchase["purchase_rate"], color=colors)
axes[1].set_title("Taxa de purchase por dia da semana (%)")
axes[1].set_ylabel("% purchases")
axes[1].set_xlabel("Dia (laranja = fim de semana)")

plt.suptitle("Sazonalidade Semanal", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 8) Data readiness para recomendação

Checklist objetivo que define se o dataset está pronto para treino e quais decisões de pré-processamento são necessárias.

In [ ]:
eligible_users = int((user_counts["n_interactions"] >= MIN_INTERACTIONS_USER).sum())
eligible_items = int(
    (df.groupby("item_id").size() >= 1).sum()
)
items_with_purchase = int(
    df[df["interaction_type"] == "purchase"]["item_id"].nunique()
)
users_with_purchase = int(
    df[df["interaction_type"] == "purchase"]["user_id"].nunique()
)

readiness_checks = pd.DataFrame({
    "check": [
        f"Usuários elegíveis (≥{MIN_INTERACTIONS_USER} interações)",
        "Itens com ao menos 1 purchase",
        "Usuários com ao menos 1 purchase",
        f"Esparsidade abaixo de 99.9%",
        "Split temporal viável (dados ordenados)",
        "Ausência de missing values",
    ],
    "resultado": [
        f"{eligible_users:,} de {n_users:,} ({eligible_users/n_users*100:.1f}%)",
        f"{items_with_purchase:,} de {n_items:,} ({items_with_purchase/n_items*100:.1f}%)",
        f"{users_with_purchase:,} de {n_users:,} ({users_with_purchase/n_users*100:.1f}%)",
        f"{sparsity:.4%}",
        f"Sim — período de {(df['timestamp'].max() - df['timestamp'].min()).days} dias",
        f"{df.isna().sum().sum()} missing values",
    ],
    "status": [
        "✓ OK" if eligible_users / n_users > 0.5 else "⚠ Atenção",
        "✓ OK" if items_with_purchase / n_items > 0.5 else "⚠ Atenção",
        "✓ OK" if users_with_purchase / n_users > 0.3 else "⚠ Atenção",
        "✓ OK" if sparsity < 0.999 else "⚠ Alta esparsidade",
        "✓ OK",
        "✓ OK" if df.isna().sum().sum() == 0 else "⚠ Atenção",
    ],
})
display(readiness_checks)

In [ ]:
# Resumo de decisões de pré-processamento
decisions = pd.DataFrame({
    "decisao": [
        "Filtro de usuários frios",
        "Pesos por tipo de evento",
        "Split treino/teste",
        "Representação do modelo",
        "Estratégia de negative sampling",
    ],
    "escolha": [
        f"Manter apenas usuários com ≥{MIN_INTERACTIONS_USER} interações",
        "view=1, add_to_cart=2, purchase=3 (sinal implícito ponderado)",
        "Temporal: 80% treino (mais antigo) / 20% teste (mais recente)",
        "Embedding-based: user_id e item_id → vetores densos aprendidos",
        "Amostrar negativos aleatoriamente (itens não interagidos pelo usuário)",
    ],
    "justificativa": [
        "Usuários frios não têm sinal suficiente para embedding confiável",
        "Purchase > add_to_cart > view em intensidade de preferência",
        "Evita data leakage — modelo só vê o passado durante treino",
        "Esparsidade alta inviabiliza factorization clássica sem regularização",
        "Estratégia padrão para feedback implícito (BPR, NCF)",
    ],
})
display(decisions)

## 9) Conclusao executiva

### O que o EDA revelou

| Dimensao | Achado | Impacto no modelo |
|---|---|---|
| **Funil** | 83.7% view / 12.4% cart / 3.9% purchase | Sinal de purchase mais rico que a base original (3%) |
| **Sazonalidade** | Fim de semana: 2.5x mais trafego + purchase rate ~3x maior | Split temporal preserva esse padrao no treino |
| **Categorias** | 5 categorias com volumes e taxas de purchase distintas | Feature direta para analise de fairness e possivelmente para o modelo |
| **Genero** | ~48% M / ~47% F / ~5% NB | Monitorar paridade de recomendacao no Model Card |
| **Cauda longa** | Top 10% dos itens concentram ~55% das interacoes | Popularity bias persiste — Coverage@K obrigatorio |
| **Esparsidade** | >97% da matriz vazia | Justifica embedding-based |
| **Qualidade** | Zero missing, zero duplicados | Pronto para pre-processamento |

### Decisoes confirmadas para a modelagem

1. **Modelo:** `EmbeddingRecommender` (PyTorch) — user e item embeddings + produto interno
2. **Split:** temporal 80/20 — treino no passado, avaliacao no futuro
3. **Sinal:** feedback implicito ponderado (view < add_to_cart < purchase)
4. **Features extras:** `category` e `user_gender` disponiveis para enriquecer analises
5. **Filtro:** usuarios com < 5 interacoes excluidos do treino
6. **Metricas:** Precision@10, Recall@10, NDCG@10, Coverage@10
7. **Baselines:** PopularityRecommender + NMF (sklearn)
8. **Fairness:** monitorar Precision@10 por genero e por categoria no Model Card

### Proximos passos

- `scripts/preprocess.py` — filtro de cold users, encoding de IDs, split temporal
- `scripts/feature_eng.py` — matriz ponderada, features de popularidade e categoria
- `scripts/train.py` — treino do `EmbeddingRecommender` com MLflow tracking
- `scripts/evaluate.py` — Precision@K, Recall@K, NDCG@K, Coverage@K